In [6]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available: ", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.10.0
Num GPUs Available:  []


In [22]:
import tensorflow as tf
import pandas as pd
import os
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.preprocessing import LabelEncoder
import glob

csv_file = 'C:/Users/Valen/Desktop/Laboratorio-2-TAA/train_curated.csv'
image_dir = 'C:/Users/Valen/Desktop/Laboratorio-2-TAA/data/preproc'
data = pd.read_csv(csv_file)

label_encoder = LabelEncoder()
data['label_index'] = label_encoder.fit_transform(data['labels'])

def load_image(file_name, label):
    file_name = file_name.numpy().decode('utf-8')
    pattern = os.path.join(image_dir, '*' + file_name + '*.png')
    image_paths = glob.glob(pattern)
    if not image_paths:
        print(f"File not found for pattern: {pattern}")
        return tf.zeros((64, 64, 1)), label
    image_path = image_paths[0]
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=1)
    image = tf.image.resize(image, [64, 64])
    return image, label

def tf_load_image(file_name, label):
    file_name = tf.cast(file_name, tf.string)
    label = tf.cast(label, tf.int32)
    image, label = tf.py_function(load_image, [file_name, label], [tf.float32, tf.int32])
    image.set_shape((64, 64, 1))
    label.set_shape([])
    return image, label

file_names = data['fname'].apply(lambda x: x.split('.')[0]).values
labels = data['label_index'].values

labels = labels.astype('int32')

dataset = tf.data.Dataset.from_tensor_slices((file_names, labels))
dataset = dataset.map(tf_load_image)

train_size = int(0.8 * len(dataset))
train_dataset = dataset.take(train_size)
test_dataset = dataset.skip(train_size)

normalization_layer = tf.keras.layers.Rescaling(1./255)
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset = test_dataset.map(lambda x, y: (normalization_layer(x), y))

BATCH_SIZE = 32
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

for image, label in train_dataset.take(1):
    print(image.shape, label.numpy())

In [ ]:
train_dataset

<PrefetchDataset element_spec=(TensorSpec(shape=(None, 64, 64, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [ ]:
import tensorflow as tf

# Normalización de los datos
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset = test_dataset.map(lambda x, y: (normalization_layer(x), y))


In [ ]:
def ConvBlock(x, filters, downsample=False):
    residual = x
    if downsample:
        residual = tf.keras.layers.Conv2D(filters, (1, 1), strides=(2, 2), padding='same', use_bias=False)(residual)
        residual = tf.keras.layers.BatchNormalization()(residual)

    x = tf.keras.layers.Conv2D(filters, (3, 3), padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(filters, (3, 3), padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    if downsample:
        x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = tf.keras.layers.Add()([x, residual])
    x = tf.keras.layers.ReLU()(x)

    return x

def create_model(input_shape, num_classes):
    inputs = tf.keras.layers.Input(shape=input_shape)
    
    x = ConvBlock(inputs, 64)
    x = ConvBlock(x, 128, downsample=True)
    x = ConvBlock(x, 256, downsample=True)
    x = ConvBlock(x, 512, downsample=True)
    
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    
    model = tf.keras.models.Model(inputs, outputs)
    return model


Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 64, 64, 1)]  0           []                               
                                                                                                  
 conv2d_22 (Conv2D)             (None, 64, 64, 64)   576         ['input_3[0][0]']                
                                                                                                  
 batch_normalization_16 (BatchN  (None, 64, 64, 64)  256         ['conv2d_22[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 re_lu_16 (ReLU)                (None, 64, 64, 64)   0           ['batch_normalization_16[0]

KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Leer el CSV
data = pd.read_csv('C:/Users/Valen/Desktop/Laboratorio-2-TAA/train_curated.csv')

# Extraer las etiquetas
file_names = data['fname'].values
labels = data['labels'].values


In [ ]:
len(labels)

4970

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Crear un codificador de etiquetas
label_encoder = LabelEncoder()

# Ajustar y transformar las etiquetas
encoded_labels = label_encoder.fit_transform(labels)

# Verificar el número de clases
num_classes = len(label_encoder.classes_)
print(f"Number of classes: {num_classes}")


Number of classes: 213


In [ ]:
input_shape = (64, 64, 1)  # Ajusta esto según tus datos (1 para imágenes en escala de grises)
model = create_model(input_shape, num_classes)
model.summary()

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

num_epochs = 10  # Ajusta el número de épocas según tu preferencia
model.fit(train_dataset, epochs=num_epochs, validation_data=test_dataset)

Epoch 1/10


InvalidArgumentError: Graph execution error:

Detected at node 'sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/SparseSoftmaxCrossEntropyWithLogits' defined at (most recent call last):
    File "c:\Users\Valen\anaconda3\envs\myenv\lib\runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "c:\Users\Valen\anaconda3\envs\myenv\lib\runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "C:\Users\Valen\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "C:\Users\Valen\AppData\Roaming\Python\Python310\site-packages\traitlets\config\application.py", line 1075, in launch_instance
      app.start()
    File "C:\Users\Valen\AppData\Roaming\Python\Python310\site-packages\ipykernel\kernelapp.py", line 739, in start
      self.io_loop.start()
    File "C:\Users\Valen\AppData\Roaming\Python\Python310\site-packages\tornado\platform\asyn

In [ ]:
import tensorflow as tf
import pandas as pd
import os
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.preprocessing import LabelEncoder
import glob

csv_file = 'C:/Users/Valen/Desktop/Laboratorio-2-TAA/train_curated.csv'
image_dir = 'C:/Users/Valen/Desktop/Laboratorio-2-TAA/data/preproc'

data = pd.read_csv(csv_file)
label_encoder = LabelEncoder()
data['label_index'] = label_encoder.fit_transform(data['labels'])

def load_image(file_name, label):
    file_name = file_name.numpy().decode('utf-8')
    pattern = os.path.join(image_dir, '*' + file_name + '*.png')
    image_paths = glob.glob(pattern)
    if not image_paths:
        print(f"File not found for pattern: {pattern}")
        return tf.zeros((64, 64, 1)), label
    image_path = image_paths[0]
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=1)
    image = tf.image.resize(image, [64, 64])
    return image, label

def tf_load_image(file_name, label):
    file_name = tf.cast(file_name, tf.string)
    label = tf.cast(label, tf.int32)
    image, label = tf.py_function(load_image, [file_name, label], [tf.float32, tf.int32])
    image.set_shape((64, 64, 1))
    label.set_shape([])
    return image, label

file_names = data['fname'].apply(lambda x: x.split('.')[0]).values
labels = data['label_index'].values

labels = labels.astype('int32')

dataset = tf.data.Dataset.from_tensor_slices((file_names, labels))
dataset = dataset.map(tf_load_image)

train_size = int(0.8 * len(dataset))
train_dataset = dataset.take(train_size)
test_dataset = dataset.skip(train_size)

normalization_layer = tf.keras.layers.Rescaling(1./255)
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset = test_dataset.map(lambda x, y: (normalization_layer(x), y))

BATCH_SIZE = 32
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

for image, label in train_dataset.take(1):
    print(image.shape, label.numpy())